# Final-token probes: binary versus ternary top rule

Train the same linear probes on mixed binary/ternary trees at checkpoints 512, 16384, 262144, and 2097152. Evaluate each fitted probe on two independent test sets: **ternary root** and **binary root**. Only the root arity is fixed; all subsequent rules follow the original mixed distribution (equal probability of binary and ternary arity, uniform within each arity).

The training data, seeds, last-input-position representation, and probe hyperparameters are unchanged. Each test set contains 32768 examples, reused across checkpoints. Results and plots use `top_rule` filenames, separate from the old pure-ternary experiment. Run from the repository root. The first run must fit every stage because the old caches contain neither fitted probes nor these test evaluations.


## 1. Setup and probe datasets

In [ ]:
import gc, sys, pickle, warnings
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

WORKSPACE = Path.cwd()
if not (WORKSPACE / 'models').exists(): raise RuntimeError('Run from the repository root.')
sys.path[:0] = [str(WORKSPACE), str(WORKSPACE / 'datasets' / 'rhm')]
import models
from mixed_random_hierarchy_model import (
    sample_padded_rules, create_probabilities,
    sample_data_from_labels_varying_tree_tensorized,
)

V, L, BIN = 8, 3, 2
M_2, M_3, D_MAX, PAD_TOKEN = BIN, BIN * V, 3 ** L, V
T = D_MAX - 1
MODEL_DEPTH = 12
STEPS = [512, 16384, 262144, 2097152]
SEED_RULES, TRAIN_SEED, TEST_SEED = 12345678, 42, 43
N_TRAIN, N_TEST = 32768, 32768
NUM_HEADS, BATCH_SIZE, PROBE_C = 16, 256, 0.1
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
CHECKPOINTS = {step: WORKSPACE / f'run_L3_v8_bin2_ter16_P16777216_lr0.001_l20.03_bs128_depth12_demb_256_nheads_16_arr0_t{step}.pt' for step in STEPS}
RESULT_PATHS = {step: WORKSPACE / f'probe_results_step_{step}_mixed_train_top_rule_test_last_token_depth12.pkl' for step in STEPS}
RECOMPUTE = False  # Reuse only this top-rule experiment's caches.
TEST_SEEDS = {'ternary': TEST_SEED, 'binary': TEST_SEED + 1}
EXPERIMENT = 'mixed_train_top_rule_test_depth12_v1'
# Cached stages need no checkpoint file.
missing = [CHECKPOINTS[step] for step in STEPS
           if (RECOMPUTE or not RESULT_PATHS[step].is_file())
           and not CHECKPOINTS[step].is_file()]
if missing: raise FileNotFoundError('Missing checkpoints:\n' + '\n'.join(map(str, missing)))

rules = sample_padded_rules(V, V, M_2, M_3, 2, 3, L, SEED_RULES)
mixed_probs = create_probabilities(M_2, M_3, L)
def top_rule_probabilities(arity):
    probabilities = {level: prob.clone() for level, prob in mixed_probs.items()}
    if arity == 'ternary':
        probabilities[0] = torch.cat([torch.zeros(M_2), torch.full((M_3,), 1/M_3)])
    elif arity == 'binary':
        probabilities[0] = torch.cat([torch.full((M_2,), 1/M_2), torch.zeros(M_3)])
    else:
        raise ValueError(arity)
    return probabilities

TEST_PROBS = {arity: top_rule_probabilities(arity) for arity in TEST_SEEDS}

def generate_sentences(n, probabilities, seed):
    state = torch.random.get_rng_state(); torch.manual_seed(seed)
    labels = torch.randint(0, V, (n,))
    sentences, _ = sample_data_from_labels_varying_tree_tensorized(labels, rules, probabilities, V, D_MAX)
    torch.random.set_rng_state(state)
    return sentences

def make_last_token_task(sentences):
    # Mirrors init.MySentenceDataLoader exactly.
    n = len(sentences); lengths = (sentences != PAD_TOKEN).sum(1)
    targets = sentences[torch.arange(n), lengths - 1]
    input_lengths = lengths - 1
    columns = torch.arange(T).unsqueeze(0).expand(n, T)
    source_columns = columns - T + input_lengths.unsqueeze(1)
    gathered = torch.gather(sentences[:, :T], 1, source_columns.clamp(min=0))
    inputs = torch.where(source_columns >= 0, gathered, torch.full_like(gathered, PAD_TOKEN))
    return inputs, targets, lengths

mixed_sentences = generate_sentences(N_TRAIN, mixed_probs, TRAIN_SEED)
X_train_tokens, y_train, mixed_lengths = make_last_token_task(mixed_sentences)
test_tasks = {}
for arity, seed in TEST_SEEDS.items():
    sentences = generate_sentences(N_TEST, TEST_PROBS[arity], seed)
    inputs, targets, lengths = make_last_token_task(sentences)
    root_children = 3 if arity == 'ternary' else 2
    assert torch.all((lengths >= root_children * 2 ** (L-1)) &
                     (lengths <= root_children * 3 ** (L-1)))
    assert torch.all(targets < V)
    test_tasks[arity] = {'inputs': inputs, 'targets': targets, 'lengths': lengths}
    print(f'{arity} root test={N_TEST:,}; lengths={lengths.min().item()}..{lengths.max().item()}')
assert torch.any(mixed_lengths < D_MAX) and torch.any(mixed_lengths == D_MAX)
assert torch.all(y_train < V)
print(f'device={DEVICE}; mixed train={N_TRAIN:,}')


## 2. Model loading, last-position activation extraction, and probing

In [ ]:
def infer_arch(state_dict):
    width = state_dict['token_embedding_table.weight'].shape[1]
    n_layers = max(int(k.split('.')[1]) for k in state_dict if k.startswith('blocks.')) + 1
    return dict(vocab_size=state_dict['token_embedding_table.weight'].shape[0],
                block_size=state_dict['blocks.0.attn.tril'].shape[0],
                embedding_dim=width, num_heads=NUM_HEADS,
                ffwd_size=state_dict['blocks.0.ffwd.net.0.weight'].shape[0] // width,
                num_layers=n_layers, dropout=0.0,
                rope='position_embedding_table.weight' not in state_dict, share_emb=False)

def load_model(path):
    saved = torch.load(path, map_location='cpu', weights_only=False)
    # Compiled checkpoints prefix parameter names with _orig_mod.
    state_dict = {key.removeprefix('_orig_mod.'): value
                  for key, value in saved['model'].items()}
    arch = infer_arch(state_dict); assert arch['block_size'] == T
    if (arch['num_layers'], arch['embedding_dim']) != (MODEL_DEPTH, 256):
        raise ValueError(f'Expected 12 layers and embedding width 256, got {arch}')
    model = models.CLM(**arch); model.load_state_dict(state_dict)
    return model.to(DEVICE).eval(), arch, int(saved.get('step', path.stem.rsplit('_t', 1)[-1]))

@torch.no_grad()
def extract_last_position_activations(model, inputs):
    # Shape: [embedding + blocks, examples, hidden width]. No other token positions are retained.
    by_depth = [[] for _ in range(model.num_layers + 1)]
    for start in range(0, len(inputs), BATCH_SIZE):
        batch = inputs[start:start + BATCH_SIZE].to(DEVICE); captured = {}
        hooks = [model.blocks[0].register_forward_pre_hook(
            lambda module, args: captured.update({0: args[0][:, -1].detach().cpu()}))]
        for depth, block in enumerate(model.blocks, 1):
            hooks.append(block.register_forward_hook(
                lambda module, args, output, depth=depth: captured.update({depth: output[:, -1].detach().cpu()})))
        model(batch)
        for hook in hooks: hook.remove()
        for depth in range(model.num_layers + 1): by_depth[depth].append(captured[depth].numpy())
    return np.stack([np.concatenate(chunks) for chunks in by_depth])

def probe_across_depth(train_acts, test_acts):
    # Fit once at each depth; use exactly that probe for both test populations.
    results = {'train_accuracy': [], 'train_cross_entropy_normalized': [],
               'tests': {arity: {'test_accuracy': [], 'test_cross_entropy_normalized': []}
                         for arity in test_tasks}}
    train_labels = y_train.numpy()
    for depth, X_train in enumerate(train_acts):
        probe = Pipeline([('scaler', StandardScaler()), ('classifier', LogisticRegression(
            C=PROBE_C, solver='lbfgs', max_iter=5000, random_state=0))])
        probe.fit(X_train, train_labels)
        results['train_accuracy'].append(probe.score(X_train, train_labels))
        results['train_cross_entropy_normalized'].append(
            log_loss(train_labels, probe.predict_proba(X_train), labels=np.arange(V)) / np.log(V))
        for arity, acts in test_acts.items():
            labels = test_tasks[arity]['targets'].numpy()
            metrics = results['tests'][arity]
            metrics['test_accuracy'].append(probe.score(acts[depth], labels))
            metrics['test_cross_entropy_normalized'].append(
                log_loss(labels, probe.predict_proba(acts[depth]), labels=np.arange(V)) / np.log(V))
            print(f"  depth {depth}, {arity} root: acc={metrics['test_accuracy'][-1]:.4f}; "
                  f"CE/log(V)={metrics['test_cross_entropy_normalized'][-1]:.4f}")
    counts = np.bincount(train_labels, minlength=V)
    results.update(chance_uniform=1/V, train_majority_baseline=counts.max()/counts.sum())
    return results

def valid_cache(payload):
    return (payload.get('experiment') == EXPERIMENT
            and payload.get('probe_C') == PROBE_C
            and payload.get('n_train_mixed') == N_TRAIN
            and payload.get('n_test_per_top_rule') == N_TEST
            and payload.get('train_seed') == TRAIN_SEED
            and payload.get('test_seeds') == TEST_SEEDS
            and all(len(payload.get('token_probe', {}).get('tests', {}).get(arity, {}).get(metric, [])) == MODEL_DEPTH + 1
                    for arity in TEST_SEEDS
                    for metric in ('test_accuracy', 'test_cross_entropy_normalized')))


## 3. Run missing checkpoints

Fit one probe per depth on the unchanged mixed training set, evaluate it on both root-conditioned test sets, and save both evaluations together. `RECOMPUTE = False` reuses valid top-rule caches only.


In [ ]:
token_probe_results = {}
architecture = None
for requested_step in STEPS:
    print(f'\n=== checkpoint t={requested_step} ===')
    if RESULT_PATHS[requested_step].exists() and not RECOMPUTE:
        with RESULT_PATHS[requested_step].open('rb') as handle: cached = pickle.load(handle)
        if valid_cache(cached) and cached.get('checkpoint_step') == requested_step:
            token_probe_results[requested_step] = cached['token_probe']
            print('reusing:', RESULT_PATHS[requested_step])
            continue
        print('incompatible cache; refitting this checkpoint')
    model, arch, loaded_step = load_model(CHECKPOINTS[requested_step])
    if loaded_step != requested_step: raise ValueError(f'Expected step {requested_step}, checkpoint reports {loaded_step}')
    if architecture is None: architecture = arch
    elif (arch['num_layers'], arch['embedding_dim']) != (architecture['num_layers'], architecture['embedding_dim']):
        raise ValueError('Checkpoint architectures differ.')
    train_acts = extract_last_position_activations(model, X_train_tokens)
    test_acts = {arity: extract_last_position_activations(model, task['inputs'])
                 for arity, task in test_tasks.items()}
    result = probe_across_depth(train_acts, test_acts)
    token_probe_results[requested_step] = result
    payload = {'token_probe': result, 'checkpoint_step': requested_step, 'probe_C': PROBE_C,
               'experiment': EXPERIMENT, 'n_train_mixed': N_TRAIN, 'n_test_per_top_rule': N_TEST,
               'train_seed': TRAIN_SEED, 'test_seeds': TEST_SEEDS,
               'root_arities': {'ternary': 3, 'binary': 2},
               'descendant_distribution': 'original mixed', 'rules_seed': SEED_RULES,
               'probe_position': 'last input position'}
    with RESULT_PATHS[requested_step].open('wb') as handle: pickle.dump(payload, handle)
    print('saved:', RESULT_PATHS[requested_step])
    del model, train_acts, test_acts; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    elif DEVICE == 'mps': torch.mps.empty_cache()

## 4. Regenerate final comparison plots from saved results

Run the next cell independently after a kernel restart. Stage colors are shared across sets; solid circles denote ternary roots and dashed squares binary roots. It exports the combined accuracy/loss figure, separate panels, and log-loss panel as PDF and PNG. Old theoretical plateaus are omitted because they have not been derived for these root-conditioned mixed distributions. The Inside/Outside divider remains at L6.

To use the later Stage-3 checkpoint, change `STEPS` in setup to `[512, 16384, 741456, 2097152]` before running Sections 1–3, then change `PLOT_STEPS` below to match. Existing caches for the other stages are reused.


In [ ]:
# Standalone plotting cell: no model loading or probe fitting.
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

WORKSPACE = Path.cwd()
PLOT_STEPS = [512, 16384, 262144, 2097152]
MODEL_DEPTH, V = 12, 8
styles = {'ternary': ('-', 'o'), 'binary': ('--', 's')}
results = {}
for step in PLOT_STEPS:
    path = WORKSPACE / f'probe_results_step_{step}_mixed_train_top_rule_test_last_token_depth12.pkl'
    if not path.is_file():
        raise FileNotFoundError(f'Run Sections 1–3 first; missing {path}')
    with path.open('rb') as handle:
        payload = pickle.load(handle)
    if payload.get('experiment') != 'mixed_train_top_rule_test_depth12_v1' or payload.get('checkpoint_step') != step:
        raise ValueError(f'Incompatible result: {path}')
    results[step] = payload['token_probe']['tests']
    for arity in styles:
        for metric in ('test_accuracy', 'test_cross_entropy_normalized'):
            values = np.asarray(results[step][arity][metric], dtype=float)
            if values.shape != (MODEL_DEPTH + 1,) or not np.isfinite(values).all():
                raise ValueError(f'Invalid {arity} {metric}: {path}')

plt.rcParams.update({'font.size': 9, 'axes.labelsize': 10, 'legend.fontsize': 8,
                     'pdf.fonttype': 42, 'ps.fonttype': 42})
cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
colors = {step: cycle[i] for i, step in enumerate(PLOT_STEPS)}
depth = np.arange(MODEL_DEPTH + 1)
layer_labels = ['emb'] + [f'L{k}' for k in range(1, MODEL_DEPTH + 1)]
legend_handles = [Line2D([], [], color=colors[step], label=f'Stage {i+1}')
                  for i, step in enumerate(PLOT_STEPS)]
legend_handles += [Line2D([], [], color='black', ls=ls, marker=marker,
                         label=f'{arity.capitalize()} root')
                   for arity, (ls, marker) in styles.items()]

def draw_panel(ax, metric, log=False):
    for step in PLOT_STEPS:
        for arity, (ls, marker) in styles.items():
            ax.plot(depth, results[step][arity][metric], color=colors[step],
                    ls=ls, marker=marker, ms=3.5, lw=1.8)
    accuracy = metric == 'test_accuracy'
    ax.axhline(1/V if accuracy else 1., color='grey', ls=':', lw=1)
    ax.axvline(6, color='black', ls='--', lw=1, alpha=.55)
    ax.text(.43, .98, 'Inside', transform=ax.transAxes, ha='right', va='top')
    ax.text(.57, .98, 'Outside', transform=ax.transAxes, ha='left', va='top')
    ax.set(xticks=depth, xticklabels=layer_labels, xlabel='Transformer depth',
           ylabel='Final-token accuracy' if accuracy else r'Cross-entropy / $\log V$')
    ax.tick_params(axis='x', labelsize=7)
    if accuracy:
        ax.set_ylim(0, 1.04)
    if log:
        ax.set_yscale('log')
    ax.spines[['top', 'right']].set_visible(False)

suffix = '_stage3_t741456' if PLOT_STEPS[2] == 741456 else ''
def export(fig, stem):
    fig.tight_layout(rect=(0, 0, 1, .87))
    for ext in ('pdf', 'png'):
        path = WORKSPACE / f'{stem}_top_rule_depth12{suffix}.{ext}'
        fig.savefig(path, dpi=300, bbox_inches='tight')
        print('saved:', path)
    plt.show()

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, metric in zip(axes, ('test_accuracy', 'test_cross_entropy_normalized')):
    draw_panel(ax, metric)
fig.legend(handles=legend_handles, loc='upper center', ncol=3, frameon=False)
export(fig, 'token_probe_accuracy_and_cross_entropy_across_depth')

for metric, stem, log in [
    ('test_accuracy', 'token_probe_accuracy_across_depth', False),
    ('test_cross_entropy_normalized', 'token_probe_cross_entropy_across_depth', False),
    ('test_cross_entropy_normalized', 'token_probe_cross_entropy_across_depth_log', True),
]:
    fig, ax = plt.subplots(figsize=(5, 3.8))
    draw_panel(ax, metric, log=log)
    fig.legend(handles=legend_handles, loc='upper center', ncol=3, frameon=False)
    export(fig, stem)
